<a href="https://colab.research.google.com/github/aiman0642/saas-retention-intelligence/blob/main/11_recommendation_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 11 — Recommendation Layer

**Project:** SaaS Customer Retention Intelligence System

**Goal of this notebook:** turn a risk score + a SHAP explanation into
something a retention team can actually *do* — a suggested next action
per account, tailored to *why* that account is flagged, not just a
generic "this account is high risk" alert.

**Important framing, stated once here and repeated in the README:** this
model predicts risk. It does not prove that any specific intervention
will prevent churn. Every output in this notebook is a **suggested
retention action**, not a guaranteed fix — that distinction is kept
explicit throughout rather than implied away.

**Assumes:** Modules 9 and 10 have been run (needs
`account_shap_factors.csv` and `account_revenue_risk.csv`).

## 11.1 Mount Google Drive & Load Risk, Revenue, and SHAP Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import re
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

PROJECT_DIR = "/content/drive/MyDrive/saas-retention-intelligence"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"

revenue_risk = pd.read_csv(f"{PROCESSED_DIR}/account_revenue_risk.csv")
shap_factors = pd.read_csv(f"{PROCESSED_DIR}/account_shap_factors.csv")

print(f"revenue_risk: {revenue_risk.shape}")
print(f"shap_factors: {shap_factors.shape}")

revenue_risk: (500, 11)
shap_factors: (500, 3)


## 11.2 Combine Into One Account-Level View

Everything a retention team needs to see about one account, in one row:
risk level, dollar exposure, and *why* the model flagged them.

In [4]:
account_actions = revenue_risk.merge(shap_factors, on="account_id", how="left")
print(f"account_actions shape: {account_actions.shape}")
account_actions[["account_id", "risk_level", "expected_mrr_at_risk", "top_risk_increasing"]].head(3)

account_actions shape: (500, 13)


,account_id,risk_level,expected_mrr_at_risk,top_risk_increasing
0,A-2e4581,Medium,281.381247,tenure_months (+0.614); tenure_days (+0.613); ...
1,A-43a9e3,Medium,391.134272,engagement_trend_ratio (+0.494); total_errors ...
2,A-0a282f,Medium,64.367110,total_usage_count (+1.291); industry_DevTools ...


## 11.3 Map the Top Risk-Increasing Factor to a Suggested Action

Module 9's `top_risk_increasing` column lists each account's top 3
SHAP-flagged factors as text (e.g. `"engagement_trend_ratio (+0.61);
total_errors (+0.48); ..."`). Extract the single strongest factor and
map it to a suggested action category — the recommendation is tailored
to *why* the account is at risk, not a one-size-fits-all message.

In [5]:
def extract_top_feature(factor_string):
    """Pull the feature name from '<feature> (+val); <feature2> (+val2)...'"""
    if pd.isna(factor_string):
        return None
    match = re.match(r"([a-zA-Z_]+)", factor_string)
    return match.group(1) if match else None

account_actions["top_feature"] = account_actions["top_risk_increasing"].apply(extract_top_feature)

# Maps a feature name to (action category, suggested action text).
# Grouped by theme rather than one rule per exact column name, so a new
# similarly-themed feature added later doesn't silently fall through.
ACTION_MAP = {
    "engagement_trend_ratio": ("Engagement", "Proactive check-in — usage has been trending down; offer a refresher demo or usage walkthrough"),
    "usage_rate_per_day": ("Engagement", "Usage education — send feature tips or a re-onboarding session to lift day-to-day usage"),
    "total_usage_count": ("Engagement", "Usage education — send feature tips or a re-onboarding session to lift day-to-day usage"),
    "distinct_features_used": ("Engagement", "Feature discovery outreach — account is using a narrow slice of the product; introduce underused features"),
    "feature_breadth_ratio": ("Engagement", "Feature discovery outreach — account is using a narrow slice of the product; introduce underused features"),
    "days_since_last_activity": ("Engagement", "Re-engagement outreach — account has gone quiet recently; a personal check-in email or call"),
    "total_errors": ("Support/Technical", "Technical support outreach — elevated error activity; proactively offer troubleshooting help"),
    "ticket_count": ("Support/Technical", "Support review — high ticket volume; escalate to a senior support rep for a resolution review"),
    "escalation_count": ("Support/Technical", "Support review — escalated tickets on file; escalate to a senior support rep for a resolution review"),
    "avg_satisfaction": ("Support/Technical", "Satisfaction recovery — low support satisfaction on record; consider a service recovery outreach"),
    "avg_resolution_hours": ("Support/Technical", "Support review — slow historical resolution times; check current open tickets for delays"),
    "tenure_days": ("Relationship", "Loyalty/retention check-in — tenure-related risk signal; a relationship-focused outreach from account management"),
    "tenure_months": ("Relationship", "Loyalty/retention check-in — tenure-related risk signal; a relationship-focused outreach from account management"),
    "mrr_amount": ("Commercial", "Pricing/value conversation — revenue-linked risk signal; review plan fit and discuss value delivered"),
    "arr_amount": ("Commercial", "Pricing/value conversation — revenue-linked risk signal; review plan fit and discuss value delivered"),
    "clv_proxy": ("Commercial", "Pricing/value conversation — revenue-linked risk signal; review plan fit and discuss value delivered"),
    "support_tickets_per_month": ("Support/Technical", "Support review — frequent support contact relative to tenure; escalate for a resolution review"),
    "industry_": ("Segment-Level", "Not an individually actionable behavior — this account's industry segment shows elevated structural risk; consider segment-level strategy (tailored onboarding or pricing for this vertical) rather than one-off outreach"),
    "country_": ("Segment-Level", "Not an individually actionable behavior — this account's region shows elevated structural risk; consider region-specific support/localization review rather than one-off outreach"),
    "referral_source_": ("Segment-Level", "Not an individually actionable behavior — this account's acquisition channel shows elevated structural risk; consider reviewing onboarding quality for this channel rather than one-off outreach"),
}

def suggest_action(feature):
    if feature is None:
        return ("Unclear", "No dominant single factor identified — review the account's full SHAP breakdown individually")
    for key, action in ACTION_MAP.items():
        if key in feature:
            return action
    return ("Other", f"Review account manually — top factor ('{feature}') has no predefined playbook yet")

account_actions[["action_category", "suggested_action"]] = account_actions["top_feature"].apply(
    lambda f: pd.Series(suggest_action(f))
)

print(account_actions["action_category"].value_counts())

action_category
Engagement           235
Segment-Level        106
Relationship         105
Support/Technical     53
Commercial             1
Name: count, dtype: int64


## 11.4 Apply Urgency Based on Risk Level

The *type* of action comes from the top SHAP factor (12.3); the
*urgency* comes from the risk tier. Combine both into one clear line —
and only generate action rows for accounts that are actually still
active and worth acting on (a churned account doesn't need outreach).

In [6]:
URGENCY_MAP = {
    "High": "Act this week",
    "Medium": "Monitor / act within the month",
    "Low": "No action needed",
}

account_actions["urgency"] = account_actions["risk_level"].map(URGENCY_MAP)

# Only active accounts need a recommendation — churned accounts are past
# the point of retention outreach.
actionable = account_actions[
    (account_actions["churn_flag"] == False) & (account_actions["risk_level"] != "Low")
].copy()

print(f"Actionable accounts (Medium/High risk, still active): {len(actionable)}")

Actionable accounts (Medium/High risk, still active): 291


## 11.5 Example: Prioritized Action List

What Module 13's dashboard would actually show a retention team —
sorted by dollar exposure (per Module 10's finding that this differs
from sorting by raw probability), with the specific reason and action
attached.

In [7]:
display_cols = ["account_id", "risk_level", "urgency", "churn_probability",
                 "expected_mrr_at_risk", "action_category", "suggested_action"]

prioritized = actionable.sort_values("expected_mrr_at_risk", ascending=False)
print(prioritized[display_cols].head(10).to_string(index=False))

account_id risk_level                        urgency  churn_probability  expected_mrr_at_risk action_category                                                                                                                                                                                                          suggested_action
  A-d4e0d4     Medium Monitor / act within the month           0.464800          10821.936998    Relationship                                                                                                          Loyalty/retention check-in — tenure-related risk signal; a relationship-focused outreach from account management
  A-d792a6       High                  Act this week           0.816664          10563.551743      Engagement                                                                                                                            Proactive check-in — usage has been trending down; offer a refresher demo or usage walkthrough
  A-0651a4     M

## 11.6 Action Category Breakdown

A quick view of what *kinds* of interventions the retention team would
be running most often, based on this account base's actual top risk
factors — useful for a business to know whether to invest more in
support capacity, onboarding, or account management.

In [8]:
category_summary = actionable.groupby("action_category").agg(
    accounts=("account_id", "count"),
    total_mrr_at_risk=("expected_mrr_at_risk", "sum"),
).sort_values("total_mrr_at_risk", ascending=False)

print(category_summary)

                   accounts  total_mrr_at_risk
action_category                               
Engagement              136      156202.386098
Relationship             76       81226.244281
Segment-Level            52       56025.073622
Support/Technical        27       18966.588524


## 11.7 Action Category — Key Findings

Based on the actual breakdown above, among the 291 actionable
(Medium/High risk, still-active) accounts:

- **Engagement issues are the largest actionable category** — 136
  accounts, $156,202 in MRR at risk — confirming Module 9's SHAP
  finding that `engagement_trend_ratio` is the single strongest global
  driver. This is genuinely good news operationally: it's the most
  addressable category (usage education, re-engagement outreach) of
  the four.
- **Relationship/tenure-linked risk is second** (76 accounts, $81,226)
  — consistent with the documented `tenure_days` limitation from
  Module 5, so this category should be read with that caveat in mind
  rather than as a clean behavioral signal.
- **Segment-Level (structural) risk affects 52 accounts and $56,025 in
  exposure** — these accounts' top SHAP factor is their industry,
  country, or referral source, not an individual behavior. This is a
  genuine, actionable business finding in its own right: some risk
  here isn't about what an individual account is doing, it's about
  which segment they belong to — worth a strategic (not individual)
  response.
- **Support/Technical is the smallest category** (27 accounts,
  $18,967) — support experience isn't the primary churn driver for
  most at-risk accounts in this dataset, which itself is useful for a
  business deciding where to invest retention resources.

## 11.8 Important Distinction — Restated Explicitly

**This model predicts risk. It does not prove that any suggested
action will prevent churn.** The `suggested_action` column is a
starting point for a human retention team's judgment, informed by
which factor the model weighted most heavily for that account — not a
validated causal intervention. No A/B test or causal analysis backs
the claim that any specific action reduces churn for these specific
accounts. State this plainly in the README rather than let the
dashboard's confident tone imply otherwise.

## 11.9 Save Recommendation Table

For Module 12's dashboard (Customer Risk lookup page).

In [9]:
account_actions.to_csv(f"{PROCESSED_DIR}/account_recommendations.csv", index=False)
print(f"Saved account_recommendations.csv ({account_actions.shape[0]} accounts) to {PROCESSED_DIR}")

Saved account_recommendations.csv (500 accounts) to /content/drive/MyDrive/saas-retention-intelligence/data/processed


## 11.10 Module 11 Summary

- Combined risk scores, revenue exposure, and SHAP top factors into one
  account-level view
- Mapped each account's single strongest SHAP risk-increasing factor to
  one of 5 themed action categories: Engagement, Relationship,
  Segment-Level, Support/Technical, Commercial — including a dedicated
  Segment-Level category for structural factors (industry, country,
  referral source) that aren't individually actionable, rather than
  dumping them in a generic "Other" bucket
- **291 actionable accounts** (Medium/High risk, still active):
  Engagement is the largest category (136 accounts, $156,202 at risk),
  confirming Module 9's SHAP finding; Segment-Level structural risk
  affects 52 accounts ($56,025) — a genuine, distinct business finding
- Combined action type (from SHAP) with urgency (from risk tier) into a
  clear, prioritized action list sorted by dollar exposure per Module
  10's finding
- Stated explicitly, more than once, that these are suggested actions
  informed by correlation, not a proven causal fix
- Saved `account_recommendations.csv` for the dashboard

**Next:** Module 12 — Streamlit Dashboard